# Trees and Boosting Questions — Why Gradient Boosting Runs Tabular ML

Tree ensembles are the "explain random forests and gradient boosting" round — still the most common production model family for tabular data even in the deep-learning era, and interviewers probe the mechanics hard. Everything below is executed: the split criteria are computed by hand, bagging's variance reduction is measured, and XGBoost/LightGBM are raced against each other on the same data.

## Q1 — Decision Tree Split Criteria: Gini vs Entropy, Computed

**Gini** = 1 − Σp² (probability a random draw is mislabeled if you label by class frequency). **Entropy** = −Σp log₂ p (bits of uncertainty). Both minimal at pure nodes, maximal at 50/50; they're near-monotone transforms of each other, so splits rarely differ — the safe sentence: *"Gini is the sklearn default because it skips the log; entropy is the information-theoretic view; in practice they agree."*

In [1]:
import numpy as np
import pandas as pd

def gini(p):
    return 1 - (p ** 2).sum()

def entropy(p):
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

node = np.array([0.5, 0.5])           # maximally mixed
pure = np.array([1.0, 0.0])           # pure
skew = np.array([0.9, 0.1])
for name, p in [("50/50", node), ("90/10", skew), ("pure", pure)]:
    print(f"{name:>6}: gini {gini(p):.3f} | entropy {entropy(p):.3f}")

# a real split: parent 10/10 -> children (8/1) and (2/9): gain math done by hand
parent = np.array([10, 10])
left, right = np.array([8, 1]), np.array([2, 9])
pp = parent / parent.sum()
wg = (left.sum() * gini(left/left.sum()) + right.sum() * gini(right/right.sum())) / parent.sum()
we = (left.sum() * entropy(left/left.sum()) + right.sum() * entropy(right/right.sum())) / parent.sum()
print(f"\nsplit gain (gini)   : {gini(pp) - wg:.4f}")
print(f"split gain (entropy): {entropy(pp) - we:.4f}  <- both say this split is good")

 50/50: gini 0.500 | entropy 1.000
 90/10: gini 0.180 | entropy 0.469
  pure: gini 0.000 | entropy -0.000

split gain (gini)   : 0.2475
split gain (entropy): 0.3973  <- both say this split is good


## Q2 — Why Do Trees Not Need Scaling? And Other Free Lunches

Splits are threshold comparisons on single features (`x_j <= t`) — monotone transforms (standardize, log) *reorder nothing* and change only the threshold value. That also buys: mixed feature types, missing-friendly variants (LightGBM learns default directions), and immunity to outliers (a threshold doesn't care how far the far side is). The costs to volunteer: **instability** (small data change → different tree; that's why ensembles exist), axis-parallel boundaries only, and no extrapolation (a tree can never predict outside the range seen in training leaves — the classic interview gotcha for tree-based time series).

## Q3 — Bagging: Variance Reduction, Measured

**Bootstrap aggregating:** train B trees on B bootstrap samples and average. Averaging B estimates of variance σ² with pairwise correlation ρ gives variance ρσ² + (1−ρ)σ²/B — the part that survives is the *shared* error. So bagging kills variance, not bias; deep (low-bias, high-variance) trees are the ideal base learner. Random forests decorrelate further by restricting each split to a random feature subset (`max_features`) — the mechanism to name: *without it, every tree leads with the same dominant feature and errors stay correlated.*

In [2]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=500, n_features=15, n_informative=5, noise=12, random_state=0)

# single deep tree: low bias, high variance (score across resamples)
single, forest = [], []
for seed in range(15):
    idx = np.random.default_rng(seed).choice(len(X), len(X), replace=True)
    oob = ~np.isin(np.arange(len(X)), idx)
    single.append(DecisionTreeRegressor(max_depth=None, random_state=seed).fit(X[idx], y[idx]).score(X[oob], y[oob]))
    forest.append(RandomForestRegressor(n_estimators=100, random_state=seed).fit(X[idx], y[idx]).score(X[oob], y[oob]))

print(f"single deep tree : mean R2 {np.mean(single):.3f} +/- {np.std(single):.3f}  <- high variance")
print(f"random forest    : mean R2 {np.mean(forest):.3f} +/- {np.std(forest):.3f}  <- variance tamed, bias unchanged-ish")
print(f"variance reduction: {1 - np.std(forest)/np.std(single):.0%}")

single deep tree : mean R2 0.691 +/- 0.038  <- high variance
random forest    : mean R2 0.871 +/- 0.017  <- variance tamed, bias unchanged-ish
variance reduction: 56%


## Q4 — Gradient Boosting: The Additive Story (Say It Exactly)

The answer that clears the bar, word for word:

> Gradient boosting builds an **additive model** F(x) = Σ fₘ(x) in stages. At stage m it fits a **weak learner (usually a shallow tree) to the negative gradient of the loss with respect to the current predictions** — for squared error that's the residuals, for other losses it's a "pseudo-residual." Each new tree predicts *where the ensemble is currently most wrong*, and the ensemble steps a learning rate η along that correction.

**Why shallow trees?** Boosting's job is bias reduction — each tree contributes a small, correct step. Deep trees reintroduce variance and overfit the pseudo-residuals. The three-way contrast to close with: *bagging = average deep trees (variance↓), boosting = sum shallow trees (bias↓), random forest = bagging + feature subsampling (variance↓↓).*

In [3]:
# 20 lines of gradient boosting, fitted on real residuals — the interview version
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
Xr = rng.uniform(-3, 3, (400, 1))
yr = np.sin(Xr[:, 0]) + rng.normal(0, 0.2, 400)

def boost(X, y, n_estimators=80, lr=0.1):
    F = np.zeros(len(y))
    trees = []
    for _ in range(n_estimators):
        resid = y - F                        # negative gradient of squared loss
        t = DecisionTreeRegressor(max_depth=3).fit(X, resid)
        F += lr * t.predict(X)
        trees.append(t)
    return trees, F

trees, F = boost(Xr, yr)
print(f"train MSE after boosting: {np.mean((yr - F)**2):.4f}")
print("first 3 trees captured:", f"{np.mean((yr - np.zeros(400))**2):.3f} -> {np.mean((yr - (trees[0].predict(Xr)*0.1))**2):.3f} initial step")

from sklearn.ensemble import GradientBoostingRegressor
sk = GradientBoostingRegressor(n_estimators=80, learning_rate=0.1, max_depth=3).fit(Xr, yr)
print(f"sklearn GBM train MSE  : {np.mean((yr - sk.predict(Xr))**2):.4f}  <- same machinery")

train MSE after boosting: 0.0263
first 3 trees captured: 0.539 -> 0.446 initial step
sklearn GBM train MSE  : 0.0263  <- same machinery


## Q5 — XGBoost vs LightGBM vs CatBoost (the 2026 Tabular Stack)

The one-line-then-detail answer for each:

- **XGBoost** — the algorithm that industrialized GBM: second-order (Newton) boosting using loss gradients *and* Hessians, explicit L1/L2 leaf regularization, shrinkage, column/row subsampling, and engineering (parallel split finding, sparsity-aware splits). The paper's framing: regularized greedy forest per stage.
- **LightGBM** — speed through two tricks: **histogram-based splitting** (bin features into ≤255 buckets; split-finding goes from O(n·features) scans to histogram increments, and histograms are reusable across the depth) and **leaf-wise growth** (grow the leaf with max gain, not level-wise — deeper asymmetry, better accuracy, needs `num_leaves` control to avoid overfitting small data).
- **CatBoost** — **ordered boosting** (each model is fit on a permutation so its correction target was computed without it — fights the *prediction shift* of using the same data to build F and correct F) plus first-class **categorical handling** (target statistics with ordering to avoid target leakage). The pick when categoricals dominate.
- **sklearn's GradientBoosting/HistGradientBoosting** — the no-extra-dependency versions; Hist* is LightGBM-style histogram and is genuinely competitive for mid-size data.

In [4]:
import time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X, y = make_classification(n_samples=20000, n_features=25, n_informative=8,
                           n_redundant=4, weights=[0.85], random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier

results = []
for name, model in [
    ("HistGB (sklearn)", HistGradientBoostingClassifier(max_iter=200, random_state=0)),
    ("XGBoost",          xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                           eval_metric="logloss", random_state=0)),
    ("LightGBM",         lgb.LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.1,
                                            verbose=-1, random_state=0)),
]:
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    dt = time.perf_counter() - t0
    auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    results.append((name, auc, dt))
    print(f"{name:>16}: AUC {auc:.4f} | fit {dt*1000:6.0f} ms")

best = max(results, key=lambda r: r[1])
print(f"\nwinner here: {best[0]} — but on 20k rows they're within noise; differences show at scale")

HistGB (sklearn): AUC 0.9728 | fit    586 ms


         XGBoost: AUC 0.9719 | fit    924 ms


        LightGBM: AUC 0.9743 | fit    886 ms

winner here: LightGBM — but on 20k rows they're within noise; differences show at scale


## Q6 — The Hyperparameters That Matter (and How to Say You Tune Them)

Boosting has many knobs; interviewers check whether you know which *few* matter and in what order:

1. **`n_estimators` + `learning_rate`** — the fundamental tradeoff (more, weaker steps generalize better). Tune with early stopping on a validation set: fix η≈0.05–0.1, let trees run until validation loss stops improving.
2. **Tree complexity** — `max_depth` (XGBoost 3–8) or `num_leaves` (LightGBM 15–63). Deeper = more interaction capture + more overfit.
3. **Subsampling** — `subsample` (rows) and `colsample_bytree` (features), 0.6–0.9: the randomization that made boosting more robust.
4. **Regularization** — `reg_alpha` (L1), `reg_lambda` (L2), `min_child_weight`/`min_data_in_leaf` (hurdle for making a leaf at all).
5. **`scale_pos_weight` / class weights** for imbalance.

**The process sentence:** *"learning rate fixed at 0.05, early stopping picks n_estimators, then a modest search over tree complexity and subsampling — I tune few things well rather than everything poorly."*

In [5]:
# early stopping, demonstrated — the n_estimators answer in code
dtrain = xgb.DMatrix(Xtr, label=ytr); dtest = xgb.DMatrix(Xte, label=yte)
bst = xgb.train({"eta": 0.1, "max_depth": 4, "eval_metric": "logloss"},
                dtrain, num_boost_round=2000,
                evals=[(dtest, "test")], early_stopping_rounds=30, verbose_eval=False)
print(f"best iteration: {bst.best_iteration} of 2000 allowed — early stopping picked it")
print(f"test logloss at best: {bst.best_score:.4f}")

best iteration: 26 of 2000 allowed — early stopping picked it
test logloss at best: 0.1510


## Q7 — Feature Importance: Three Kinds, Three Different Answers

*Gain* (total loss reduction credited to the feature — the default; biased toward high-cardinality features), *weight/split count* (how often a feature is used — biased toward continuous features with many split points), and *SHAP* (game-theoretic marginal contributions — consistent, locally faithful, the modern answer). Always volunteer: *gain overstates correlated features because whoever splits first takes the credit; permutation importance on held-out data and SHAP are the honest versions.*

In [6]:
import xgboost as xgb2
clf = xgb2.XGBClassifier(n_estimators=150, max_depth=5, eval_metric="logloss",
                         random_state=0).fit(Xtr, ytr)

imp = pd.Series(clf.feature_importances_, index=[f"f{i}" for i in range(X.shape[1])])
print("top 5 by gain:")
print(imp.sort_values(ascending=False).head(5).round(4).to_string())

from sklearn.inspection import permutation_importance

def auc_scorer(estimator, X, y):                      # plain callable: immune to scorer API changes
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(y, estimator.predict_proba(X)[:, 1])

pi = permutation_importance(clf, Xte, yte, scoring=auc_scorer, n_repeats=5, random_state=0)
perm = pd.Series(pi.importances_mean, index=[f"f{i}" for i in range(X.shape[1])])
print("\ntop 5 by permutation (held-out, honest):")
print(perm.sort_values(ascending=False).head(5).round(4).to_string())
print("\nnote the orderings differ — gain credits whoever splits first among correlated features")

top 5 by gain:
f12    0.1859
f23    0.1596
f4     0.1276
f8     0.0865
f16    0.0825



top 5 by permutation (held-out, honest):
f12    0.0419
f4     0.0220
f17    0.0204
f8     0.0181
f1     0.0104

note the orderings differ — gain credits whoever splits first among correlated features


## Q8 — Quick-Fire Tree Round

| Question | The 15-second answer |
| --- | --- |
| Why bag deep trees but boost shallow ones? | Bagging averages away variance (deep = high variance); boosting sums bias-reducing steps (shallow = high bias, low variance per step) |
| OOB score? | Out-of-bag: each tree's unused ~37% rows act as a free validation set — RF validation without a held-out split |
| Why can't trees extrapolate? | A prediction is the mean of a leaf — no leaf contains target values outside the training range; trends need linear/models on residuals |
| `max_features` default? | Classification: √p; regression: p (all) — the sklearn defaults; lowering it decorrelates trees |
| Boosting with early stopping — why not grid-search n_estimators? | Early stopping *is* the data-driven n_estimators; grid searching it is redundant and leaks validation |
| Is Random Forest boosting or bagging? | Bagging + random feature subsets; strictly variance reduction, trees grow in parallel — vs boosting's sequential bias correction |
| When do you NOT use gradient boosting? | Tiny data (overfits), need extrapolation, need calibrated probabilities without extra work, or high-dimensional sparse (linear/NN often better) |
| Monotonic constraints? | Tell the boosted trees a feature may only increase the prediction — domain knowledge as regularization, supported by XGBoost/LightGBM |